# FL-IDS Benchmark — Experiment 03
## Source-Only Threshold Transfer, Calibration, and Feature Shortcut Audit

This standalone notebook builds on the fixed leave-one-source-dataset-out protocol of Experiment 02, using **NF-UQ-NIDS-v2**. It trains **five methods** over **four target domains**, **five seeds**, and **three independently retrained feature variants** (**300 main model runs**). For each model it evaluates three FPR targets and three threshold strategies without additional model training.

**Methodological guardrails**
- The entire held-out target source dataset is excluded from training, source validation, and source-only threshold selection.
- The **pooled-source threshold** and **worst-source threshold** are deployable using source labels only. The **target oracle** uses held-out target labels and is strictly a *non-deployable post hoc diagnostic*.
- Feature-shift KS statistics and target calibration scores are computed **after** fitting for audit only; they are not used to select models, features, or thresholds.
- A source-only Platt map is fitted to source validation scores for a calibration diagnostic; it is not advertised as an independently validated calibration improvement.
- The three feature variants are `all` (41 NetFlow features), `no_ports`, and `no_ports_ttl` (removing `MIN_TTL` and `MAX_TTL` as well).
- The test distribution is sampled with balanced benign/attack classes. This is not production prevalence; there is no claim of capture-level independence.

**Prerequisite:** Attach the NF-UQ-NIDS-v2 dataset through Kaggle **Add Input**. This notebook does not substitute Edge-IIoTset. The audit will fail early with a clear diagnostic if NetFlow V2 columns are missing.

All notebook outputs are written under `/kaggle/working/FLIDS_Experiment03_Results/` and packaged as `/kaggle/working/FLIDS_Experiment03_Results.zip`.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'

for package in ('numpy','pandas','torch','scipy','scikit-learn','lightgbm'):
    module = 'sklearn' if package == 'scikit-learn' else package
    if importlib.util.find_spec(module) is None:
        print('Installing missing package:', package, flush=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

root = Path(os.environ.get('FLIDS_EXP03_ROOT', '/kaggle/working'))
root.mkdir(parents=True, exist_ok=True)
output = root / 'FLIDS_Experiment03_Results'
output.mkdir(exist_ok=True)
engine_path = root / 'flids_experiment03_engine.py'
print('Python:', sys.version.split()[0])
print('Output root:', root)
print('NF_UQ_DATA override:', os.environ.get('NF_UQ_DATA', '(not set)'))

In [ ]:
# Embedded English-only research engine (no separate Python file required).
engine_source = '"""Experiment 03: source-only threshold transfer, calibration, and feature shortcut audit.\nStrict leave-one-source-dataset-out NetFlow V2 protocol. Target labels are used only\nfor final evaluation and explicitly marked non-deployable oracle diagnostics.\n"""\nimport os\nfor k in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\',\'NUMEXPR_NUM_THREADS\'):os.environ[k]=\'1\'\nimport argparse,csv,hashlib,json,time,traceback,zipfile,faulthandler,random\nfrom pathlib import Path\nfrom collections import Counter,defaultdict\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom sklearn.metrics import roc_auc_score,average_precision_score\nfaulthandler.enable(all_threads=True)\nROOT=Path(os.getenv(\'FLIDS_EXP03_ROOT\',\'/kaggle/working\' if Path(\'/kaggle/working\').exists() else \'/mnt/data/exp02_local\'))\nOUT=ROOT/\'FLIDS_Experiment03_Results\';OUT.mkdir(parents=True,exist_ok=True)\nCACHE=ROOT/\'flids_exp03_cache.npz\'\nDOMAINS=[\'NF-UNSW-NB15\',\'NF-ToN-IoT\',\'NF-BoT-IoT\',\'NF-CSE-CIC-IDS2018\']\nFEATURES=[\'L4_SRC_PORT\',\'L4_DST_PORT\',\'PROTOCOL\',\'L7_PROTO\',\'IN_BYTES\',\'OUT_BYTES\',\'IN_PKTS\',\'OUT_PKTS\',\'FLOW_DURATION_MILLISECONDS\',\'TCP_FLAGS\',\'CLIENT_TCP_FLAGS\',\'SERVER_TCP_FLAGS\',\'DURATION_IN\',\'DURATION_OUT\',\'MIN_TTL\',\'MAX_TTL\',\'LONGEST_FLOW_PKT\',\'SHORTEST_FLOW_PKT\',\'MIN_IP_PKT_LEN\',\'MAX_IP_PKT_LEN\',\'SRC_TO_DST_SECOND_BYTES\',\'DST_TO_SRC_SECOND_BYTES\',\'RETRANSMITTED_IN_BYTES\',\'RETRANSMITTED_IN_PKTS\',\'RETRANSMITTED_OUT_BYTES\',\'RETRANSMITTED_OUT_PKTS\',\'SRC_TO_DST_AVG_THROUGHPUT\',\'DST_TO_SRC_AVG_THROUGHPUT\',\'NUM_PKTS_UP_TO_128_BYTES\',\'NUM_PKTS_128_TO_256_BYTES\',\'NUM_PKTS_256_TO_512_BYTES\',\'NUM_PKTS_512_TO_1024_BYTES\',\'NUM_PKTS_1024_TO_1514_BYTES\',\'TCP_WIN_MAX_IN\',\'TCP_WIN_MAX_OUT\',\'ICMP_TYPE\',\'ICMP_IPV4_TYPE\',\'DNS_QUERY_ID\',\'DNS_QUERY_TYPE\',\'DNS_TTL_ANSWER\',\'FTP_COMMAND_RET_CODE\']\nMETHODS=[\'PooledMLP\',\'PooledLightGBM\',\'LocalMLP\',\'FedAvgMLP\',\'FedProxMLP\']\nFEATURE_VARIANTS={\'all\':[], \'no_ports\':[\'L4_SRC_PORT\',\'L4_DST_PORT\'], \'no_ports_ttl\':[\'L4_SRC_PORT\',\'L4_DST_PORT\',\'MIN_TTL\',\'MAX_TTL\']}\nFEATURE_VARIANTS={k:[f for f in FEATURES if f not in drop] for k,drop in FEATURE_VARIANTS.items()}\nCFG=dict(seeds=[0,42,3407,123,2026],domains=DOMAINS,methods=METHODS,features=FEATURES,source_clients=3,rounds=30,local_epochs=1,batch_size=256,client_round_cap=512,lr=.001,prox_mu=.01,target_fprs=[.001,.01,.05],minimum_benign_for_001=1000,max_rows_per_domain_class=12000,source_validation_fraction=.20,lightgbm_trees=130,archive=\'FLIDS_Experiment03_Results.zip\',schema=\'UQ NetFlow V2; do not substitute Edge-IIoTset\',feature_variants=FEATURE_VARIANTS,risk_notes=[\'Three different source datasets are FL clients, not actual organizational deployments\',\'Stratified reservoir changes prevalence and reported precision/AUPRC\',\'No target-domain examples or labels used in training, scaling, model selection, or thresholds\',\'Source-only validation thresholds do not guarantee the same target FPR\',\'Oracle target-label thresholds are retrospective upper-bound diagnostics, never deployable methods\',\'Feature distribution shift statistics use target inputs post hoc only and never inform fitting or thresholding\',\'LocalMLP means mean score ensemble of three separately trained domain models\',\'Fixed score log transform is used, no target-fitted normalization\',\'Monotonic score calibration cannot repair AUROC ranking and is not applied to threshold selection\',\'Full numeric feature fingerprints are a row-overlap proxy, not session/capture independence\',\'Centralized baselines require data pooling; recorded FL bytes omit network overhead\'])\nif os.getenv(\'FLIDS_EXP03_SMOKE\')==\'1\':CFG.update(seeds=[0],domains=DOMAINS[:2],rounds=2,client_round_cap=128,max_rows_per_domain_class=220,lightgbm_trees=18,minimum_benign_for_001=1000)\nif os.getenv(\'FLIDS_EXP03_SEEDS\'):CFG[\'seeds\']=[int(z) for z in os.environ[\'FLIDS_EXP03_SEEDS\'].split(\',\')]\n\ndef save_json(path,obj):Path(path).write_text(json.dumps(obj,indent=2,default=str),encoding=\'utf-8\')\ndef safe_header(path):\n with open(path,\'r\',encoding=\'utf-8-sig\',errors=\'replace\',newline=\'\') as f:return next(csv.reader(f))\ndef normalized(t):return \'\'.join(c for c in str(t).lower() if c.isalnum())\ndef canonical_domain(t):\n s=normalized(t)\n if \'unsw\' in s:return DOMAINS[0]\n if \'ton\' in s and \'iot\' in s:return DOMAINS[1]\n if \'bot\' in s and \'iot\' in s:return DOMAINS[2]\n if \'cse\' in s or \'cicids2018\' in s:return DOMAINS[3]\n return None\n\ndef discover():\n roots=[Path(\'/kaggle/input\'),ROOT/\'input\']\n if os.getenv(\'NF_UQ_DATA\'):roots.insert(0,Path(os.environ[\'NF_UQ_DATA\']))\n found=[];diagnostics=[]\n for root in roots:\n  if not root.exists():continue\n  paths=[root] if root.is_file() else list(root.rglob(\'*.csv\'))\n  for p in paths:\n   try:\n    head=safe_header(p);cols=set(head);has_features=set(FEATURES).issubset(cols)\n    has_label=bool({\'Label\',\'label\'}&cols)\n    has_domain=\'Dataset\' in cols or canonical_domain(p.name) is not None\n    diagnostics.append(dict(path=str(p),columns=len(cols),valid_features=has_features,valid_label=has_label,domain_source=has_domain,missing_features=sorted(set(FEATURES)-cols)[:12]))\n    if has_features and has_label and has_domain:found.append((p,head))\n   except Exception as e:diagnostics.append(dict(path=str(p),error=str(e)[:180]))\n save_json(OUT/\'input_discovery.json\',dict(files=diagnostics,valid_candidates=[str(p) for p,_ in found]))\n combined=[v for v in found if normalized(v[0].name).startswith(\'nfuqnidsv2\')]\n if combined:return combined[:1]\n selected=[];used=set()\n for p,h in sorted(found,key=lambda v:str(v[0])):\n  domain=canonical_domain(p.name)\n  if domain and domain not in used:selected.append((p,h));used.add(domain)\n if len(selected)==4:return selected\n if found and any(\'Dataset\' in h for _,h in found):return [next(v for v in found if \'Dataset\' in v[1])]\n return selected\n\ndef synthetic_fixture():\n p=ROOT/\'synthetic_nf_exp03_SOFTWARE_TEST_ONLY.csv\';ROOT.mkdir(parents=True,exist_ok=True)\n rng=np.random.default_rng(51)\n with p.open(\'w\',newline=\'\') as f:\n  w=csv.writer(f);w.writerow(FEATURES+[\'Label\',\'Attack\',\'Dataset\'])\n  for d,domain in enumerate(DOMAINS):\n   for y in (0,1):\n    for i in range(260):\n     a=rng.gamma(2,1,len(FEATURES));a[0]=100+d*35+y*4;a[1]=200+d*11;a[4]+=y*(1.4+d*.2);a[5]+=y*.6;a[6]+=y*.8\n     w.writerow([f\'{v:.6f}\' for v in a]+[y,\'Attack\' if y else \'Benign\',domain+\'-v2\'])\n return p\n\ndef prepare():\n save_json(OUT/\'experiment_config.json\',CFG)\n if os.getenv(\'FLIDS_EXP03_SMOKE\')==\'1\':p=synthetic_fixture();found=[(p,safe_header(p))]\n else:found=discover()\n if not found:\n  raise FileNotFoundError(\'NF_UQ_NETFLOW_V2_NOT_FOUND: Attach the combined NF-UQ-NIDS-v2.csv or all four NF-v2 files in Kaggle Add Input; see input_discovery.json. Edge-IIoTset cannot be substituted. Set NF_UQ_DATA to a local CSV path if needed.\')\n rng=np.random.default_rng(2077);reservoir=defaultdict(list);seen=Counter();invalid=Counter();files_report=[]\n csv.field_size_limit(10_000_000)\n for path,head in found:\n  col={name:i for i,name in enumerate(head)}\n  lcol=col.get(\'Label\',col.get(\'label\'))\n  dcol=col.get(\'Dataset\');acol=col.get(\'Attack\')\n  fixed=canonical_domain(path.name)\n  good=bad=0\n  with path.open(\'r\',encoding=\'utf-8-sig\',errors=\'replace\',newline=\'\',buffering=1024*1024) as f:\n   reader=csv.reader(f);next(reader)\n   for row in reader:\n    if len(row)!=len(head):bad+=1;continue\n    domain=canonical_domain(row[dcol]) if dcol is not None else fixed\n    if domain not in DOMAINS:invalid[\'unknown_domain\']+=1;continue\n    label=normalized(row[lcol])\n    if label in (\'0\',\'normal\',\'benign\'):y=0\n    elif label in (\'1\',\'attack\',\'malicious\'):y=1\n    else:invalid[\'invalid_binary_label\']+=1;continue\n    try:\n     vals=np.array([float(row[col[c]]) if row[col[c]].strip() else 0. for c in FEATURES],dtype=np.float32)\n     if not np.isfinite(vals).all():invalid[\'nonfinite_values_replaced\']+=1;vals=np.nan_to_num(vals,nan=0.,posinf=0.,neginf=0.)\n    except Exception:bad+=1;continue\n    key=(domain,y);seen[key]+=1;n=seen[key]\n    item=(vals,str(row[acol]) if acol is not None else \'\')\n    buf=reservoir[key]\n    if len(buf)<CFG[\'max_rows_per_domain_class\']:buf.append(item)\n    else:\n     j=int(rng.integers(0,n))\n     if j<len(buf):buf[j]=item\n    good+=1\n   \n  files_report.append(dict(path=str(path),valid_rows=good,malformed_or_invalid_numeric=bad))\n  print(\'SCANNED\',path.name,good,\'invalid\',bad,flush=True)\n x=[];y=[];d=[];attack=[]\n for (domain,label),rows in sorted(reservoir.items()):\n  for a,t in rows:x.append(a);y.append(label);d.append(domain);attack.append(t)\n if not x:raise RuntimeError(\'NO_VALID_SOURCE_ROWS\')\n x=np.asarray(x,dtype=np.float32);y=np.asarray(y,dtype=np.int8);d=np.asarray(d);attack=np.asarray(attack)\n cov={f\'{domain}|{label}\':int(( (d==domain)&(y==label)).sum()) for domain in DOMAINS for label in (0,1)}\n min_count=100 if os.getenv(\'FLIDS_EXP03_SMOKE\')!=\'1\' else 50\n lacking=[k for k,v in cov.items() if v<min_count]\n save_json(OUT/\'data_coverage.json\',dict(pooled_rows=len(x),domain_class_counts=cov,minimum_samples_per_class=min_count,valid=not lacking,missing_or_small=lacking))\n if lacking:raise RuntimeError(\'DOMAIN_CLASS_COVERAGE_FAILED \'+str(lacking))\n np.savez_compressed(CACHE,x=x,y=y,d=d,attack=attack)\n # Exact numeric feature rows are used only as a *proxy* for duplicate overlap.\n fingerprints=np.asarray([hashlib.blake2b(np.ascontiguousarray(row).tobytes(),digest_size=12).hexdigest() for row in x])\n np.save(ROOT/\'flids_exp03_fingerprints.npy\',fingerprints)\n collision_groups=0;cross_domain_groups=0\n uniq,inv=np.unique(fingerprints,return_inverse=True)\n groups=defaultdict(set)\n for idx,k in enumerate(inv):groups[int(k)].add(str(d[idx]))\n cross_domain_groups=sum(len(v)>1 for v in groups.values())\n save_json(OUT/\'dataset_audit.json\',dict(original_counts={f\'{a}|{b}\':int(v) for (a,b),v in seen.items()},sample_counts=cov,source_files=files_report,invalid=dict(invalid),source_rows_scanned=sum(r[\'valid_rows\'] for r in files_report),sampled_rows=len(x),unique_numeric_fingerprints=len(uniq),duplicate_extra_rows=len(x)-len(uniq),cross_domain_fingerprint_groups=cross_domain_groups,duplicate_scope=\'capped sample, not full-source dedup\',split_independence=\'source dataset + exact feature fingerprint grouping for source train/validation; not true session-level independence\',features=FEATURES,sampling=\'uniform per domain and binary class; prevalence intentionally altered\',source_domain_definition=\'four separately collected sources within common NF-UQ NetFlow-v2 schema\'))\n manifests=[]\n for held in DOMAINS:\n  for seed in CFG[\'seeds\']:\n   tr,va,te=split_indices(y,d,fingerprints,held,seed)\n   for role,ids in [(\'train\',tr),(\'validation\',va),(\'test\',te)]:\n    manifests.append(dict(holdout=held,seed=seed,role=role,n=len(ids),benign=int(np.sum(y[ids]==0)),attack=int(np.sum(y[ids]==1)),index_digest=digest(ids),source_domains=\';\'.join(sorted(set(d[ids])))))\n pd.DataFrame(manifests).to_csv(OUT/\'split_manifest.csv\',index=False)\n print(\'EXP03_PREPARED\',len(x),\'rows, domain counts\',cov,\'folds\',len(CFG[\'seeds\'])*len(DOMAINS),flush=True)\n\ndef digest(ids):return hashlib.sha256(np.sort(np.asarray(ids,dtype=np.int64)).tobytes()).hexdigest()\ndef split_indices(y,d,fp,held,seed):\n tr=[];va=[];te=np.flatnonzero(d==held)\n for domain in DOMAINS:\n  if domain==held:continue\n  ids=np.flatnonzero(d==domain)\n  groups=fp[ids];unique=np.unique(groups)\n  if len(unique)<10:raise RuntimeError(f\'INSUFFICIENT_UNIQUE_GROUPS {domain}\')\n  a,b=next(GroupShuffleSplit(n_splits=1,test_size=CFG[\'source_validation_fraction\'],random_state=seed).split(ids,y[ids],groups))\n  tr.extend(ids[a]);va.extend(ids[b])\n tr=np.asarray(tr,dtype=np.int64);va=np.asarray(va,dtype=np.int64)\n assert not np.intersect1d(tr,va).size and not np.intersect1d(tr,te).size\n assert set(d[tr])==set(DOMAINS)-{held} and set(d[te])=={held}\n # Numeric fingerprint group overlap across *source* domains may occur; report, do not hide.\n for role,ids in [(\'train\',tr),(\'validation\',va),(\'test\',te)]:\n  if len(set(y[ids]))!=2:raise RuntimeError(f\'MISSING_BINARY_CLASS {held} {seed} {role}\')\n return tr,va,te\n\ndef transform(raw):\n # Fixed, source-agnostic signed log transform; no statistics or target data fitted.\n x=np.asarray(raw,dtype=np.float64)\n return np.clip(np.sign(x)*np.log1p(np.abs(np.clip(x,-1e12,1e12))),-18,18).astype(np.float32)\n\ndef threshold(scores,target):\n scores=np.asarray(scores,dtype=np.float64)\n if not scores.size or not np.isfinite(scores).all():raise RuntimeError(\'BAD_BENIGN_VALIDATION_SCORES\')\n order=np.sort(scores);k=int(np.floor(target*len(scores)))\n pivot=order[-k] if k else order[-1]\n t=float(np.nextafter(pivot,np.inf))\n actual=float(np.mean(scores>=t))\n if actual>target+1e-12:raise RuntimeError(\'THRESHOLD_INVARIANT_BROKEN\')\n return t,actual\n\ndef seedall(s):random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.set_num_threads(1)\nclass MLP(nn.Module):\n def __init__(self,n):super().__init__();self.net=nn.Sequential(nn.Linear(n,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))\n def forward(self,x):return self.net(x).squeeze(-1)\n\ndef local_train(initial,x,y,idx,seed,steps,device,prox=False,global_state=None):\n model=MLP(x.shape[1]);model.load_state_dict(initial);model.to(device);model.train()\n opt=torch.optim.Adam(model.parameters(),lr=CFG[\'lr\'])\n rng=np.random.default_rng(seed);global_weights=[w.detach().to(device) for w in initial.values()]\n for _ in range(steps):\n  batch=rng.choice(idx,size=min(CFG[\'batch_size\'],len(idx)),replace=len(idx)<CFG[\'batch_size\'])\n  xb=torch.as_tensor(x[batch],device=device);yb=torch.as_tensor(y[batch],device=device,dtype=torch.float32)\n  opt.zero_grad(set_to_none=True)\n  loss=F.binary_cross_entropy_with_logits(model(xb),yb)\n  if prox:\n   penalty=sum((p-g).square().sum() for p,g in zip(model.parameters(),global_weights))\n   loss=loss+CFG[\'prox_mu\']*.5*penalty\n  loss.backward();opt.step()\n return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n\n@torch.no_grad()\ndef predict_model(state,x,idx,device):\n m=MLP(x.shape[1]);m.load_state_dict(state);m.to(device).eval();scores=[]\n for chunk in np.array_split(idx,max(1,int(np.ceil(len(idx)/4096)))):\n  if len(chunk):scores.append(torch.sigmoid(m(torch.as_tensor(x[chunk],device=device))).cpu().numpy())\n return np.concatenate(scores).astype(np.float64)\n\ndef fit_method(method,x,y,d,train,val,test,seed,rounds,device):\n tic=time.perf_counter();communication=0;steps=0;local_stats=[]\n if method==\'PooledLightGBM\':\n  from lightgbm import LGBMClassifier\n  model=LGBMClassifier(n_estimators=CFG[\'lightgbm_trees\'],num_leaves=15,max_depth=5,learning_rate=.05,min_child_samples=15,reg_lambda=1,verbosity=-1,n_jobs=1,random_state=seed)\n  model.fit(x[train],y[train]);sv=model.predict_proba(x[val])[:,1];st=model.predict_proba(x[test])[:,1];steps=CFG[\'lightgbm_trees\']\n else:\n  seedall(seed);m=MLP(x.shape[1]);base={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}\n  params=sum(w.numel() for w in m.parameters())\n  clients=[train[d[train]==domain] for domain in DOMAINS if (d[train]==domain).any()]\n  if len(clients)!=3 or min(map(len,clients))<10:raise RuntimeError(\'CLIENT_DOMAIN_ALLOCATION_INVALID\')\n  step_per_client=[max(1,int(np.ceil(min(len(ids),CFG[\'client_round_cap\'])/CFG[\'batch_size\'])))*CFG[\'local_epochs\'] for ids in clients]\n  if method==\'PooledMLP\':\n   s=base\n   for r in range(rounds):\n    for c,stepcount in enumerate(step_per_client):\n     s=local_train(s,x,y,train,seed+r*300+c*79,stepcount,device)\n     steps+=stepcount\n   sv=predict_model(s,x,val,device);st=predict_model(s,x,test,device)\n  elif method==\'LocalMLP\':\n   vals=[];tests=[]\n   for c,ids in enumerate(clients):\n    s=base\n    for r in range(rounds):\n     s=local_train(s,x,y,ids,seed+r*300+c*79,step_per_client[c],device)\n     steps+=step_per_client[c]\n    pv=predict_model(s,x,val,device);pt=predict_model(s,x,test,device)\n    vals.append(pv);tests.append(pt)\n    local_stats.append(dict(client=c,source_domain=str(d[ids[0]]),train_samples=len(ids),auroc=float(roc_auc_score(y[test],pt)),auprc=float(average_precision_score(y[test],pt))))\n   sv=np.mean(vals,axis=0);st=np.mean(tests,axis=0)\n  else:\n   s=base\n   for r in range(rounds):\n    states=[];sizes=[]\n    for c,ids in enumerate(clients):\n     states.append(local_train(s,x,y,ids,seed+r*300+c*79,step_per_client[c],device,prox=(method==\'FedProxMLP\')))\n     sizes.append(len(ids));steps+=step_per_client[c]\n    weights=np.asarray(sizes,dtype=float);weights/=weights.sum()\n    s={name:sum(item[name]*float(w) for item,w in zip(states,weights)) for name in s}\n    communication+=2*len(clients)*params*4\n   sv=predict_model(s,x,val,device);st=predict_model(s,x,test,device)\n return np.asarray(sv,dtype=np.float64),np.asarray(st,dtype=np.float64),dict(train_seconds=time.perf_counter()-tic,optimizer_steps=steps,communication_bytes=communication,local_clients=local_stats)\n\ndef oracle_threshold(scores, labels, target):\n    """NON-DEPLOYABLE: uses held-out target labels for retrospective comparison only."""\n    neg=np.asarray(scores,dtype=np.float64)[np.asarray(labels)==0]\n    if len(neg)<2:return float(\'inf\'),float(\'nan\')\n    return threshold(neg,target)\n\ndef calc_scores(labels,scores,t):\n    labels=np.asarray(labels); pred=np.asarray(scores,dtype=np.float64)>=float(t)\n    tp=int(np.sum(pred&(labels==1))); fp=int(np.sum(pred&(labels==0)))\n    tn=int(np.sum(~pred&(labels==0))); fn=int(np.sum(~pred&(labels==1)))\n    return dict(attack_recall=tp/max(1,tp+fn),test_fpr=fp/max(1,fp+tn),\n                precision=tp/max(1,tp+fp),attack_f1=2*tp/max(1,2*tp+fp+fn),tp=tp,fp=fp,tn=tn,fn=fn)\n\ndef source_domain_threshold(source_scores, source_labels, source_domains, target):\n    """Conservative single deployment threshold from per-source benign quantiles."""\n    ts=[]\n    for domain in sorted(set(source_domains)):\n        mask=(source_domains==domain)&(source_labels==0)\n        if np.count_nonzero(mask)<30:raise RuntimeError(\'INSUFFICIENT_SOURCE_BENIGN_FOR_DOMAIN_THRESHOLD \'+str(domain))\n        ts.append(threshold(source_scores[mask],target)[0])\n    return float(max(ts)),{str(d):float(t) for d,t in zip(sorted(set(source_domains)),ts)}\n\ndef score_distribution(scores, labels):\n    scores=np.asarray(scores,dtype=np.float64);labels=np.asarray(labels)\n    out={}\n    for v,name in ((0,\'benign\'),(1,\'attack\')):\n        z=scores[labels==v]\n        if z.size:\n            out.update({f\'{name}_score_p{q}\':float(np.percentile(z,q)) for q in (5,50,95)})\n            out[f\'{name}_score_mean\']=float(np.mean(z))\n    return out\n\ndef feature_shift_diagnostics(train_x,test_x,names,seed,holdout):\n    """Post hoc unlabelled target feature shift only; no model tuning."""\n    from scipy.stats import ks_2samp\n    rng=np.random.default_rng(seed+187)\n    a=train_x[rng.choice(len(train_x),min(4000,len(train_x)),replace=False)]\n    b=test_x[rng.choice(len(test_x),min(4000,len(test_x)),replace=False)]\n    out=[]\n    for j,name in enumerate(names):\n        ks=float(ks_2samp(a[:,j],b[:,j],method=\'asymp\').statistic)\n        out.append(dict(holdout=holdout,seed=seed,feature=name,ks_statistic=ks,\n                        source_median=float(np.median(a[:,j])),target_median=float(np.median(b[:,j])),\n                        interpretation=\'posthoc_unlabelled_target_shift_not_used_to_train\'))\n    return out\n\ndef calibration_metrics(labels, probs, bins=15):\n    labels=np.asarray(labels,dtype=np.float64)\n    probs=np.clip(np.asarray(probs,dtype=np.float64),0.0,1.0)\n    brier=float(np.mean((probs-labels)**2))\n    cuts=np.minimum(np.floor(probs*bins).astype(np.int32),bins-1)\n    ece=0.0\n    for b in range(bins):\n        group=cuts==b\n        if np.any(group):\n            ece+=float(np.mean(group))*abs(float(np.mean(labels[group]))-float(np.mean(probs[group])))\n    return dict(brier=brier,ece=float(ece))\n\ndef source_only_platt(val_scores,val_labels,test_scores):\n    from sklearn.linear_model import LogisticRegression\n    vals=np.clip(np.asarray(val_scores,dtype=np.float64),1e-6,1-1e-6)\n    tests=np.clip(np.asarray(test_scores,dtype=np.float64),1e-6,1-1e-6)\n    val_logit=np.log(vals/(1-vals)).reshape(-1,1)\n    test_logit=np.log(tests/(1-tests)).reshape(-1,1)\n    # Fit only with source-validation labels, never target labels.\n    model=LogisticRegression(C=1.0,max_iter=300,random_state=0)\n    model.fit(val_logit,val_labels)\n    return model.predict_proba(val_logit)[:,1],model.predict_proba(test_logit)[:,1]\n\ndef tasks(pilot=False):\n ds=CFG[\'domains\'][:1] if pilot else DOMAINS\n ss=CFG[\'seeds\'][:1] if pilot else CFG[\'seeds\']\n ms=[\'PooledLightGBM\',\'FedAvgMLP\'] if pilot else METHODS\n variants=[\'all\'] if pilot else list(FEATURE_VARIANTS)\n return [(a,b,c,v) for a in ds for b in ss for c in ms for v in variants]\n\ndef worker(mode,worker_id,nworkers):\n data=np.load(CACHE);raw=data[\'x\'];full_x=transform(raw);y=data[\'y\'];d=data[\'d\'];fp=np.load(ROOT/\'flids_exp03_fingerprints.npy\')\n pending=[task for i,task in enumerate(tasks(mode==\'pilot\')) if i%nworkers==worker_id]\n errors=[];device=f\'cuda:{worker_id%torch.cuda.device_count()}\' if torch.cuda.is_available() else \'cpu\'\n for ix,(hold,seed,method,variant) in enumerate(pending,1):\n  key=f\'{mode}__{normalized(hold)}__{seed}__{method}__{variant}\'\n  result_path=OUT/\'records\'/f\'{key}.json\';pred_path=OUT/\'predictions_raw\'/f\'{key}.npz\'\n  if result_path.exists() and pred_path.exists():print(\'RESUME_SKIP\',key,flush=True);continue\n  try:\n   tr,va,te=split_indices(y,d,fp,hold,seed)\n   names=FEATURE_VARIANTS[variant];col=[FEATURES.index(z) for z in names];x=full_x[:,col]\n   sv,st,meta=fit_method(method,x,y,d,tr,va,te,seed,(2 if mode==\'pilot\' else CFG[\'rounds\']),device)\n   label=y[te];source_labels=y[va];source_domains=d[va];neg=sv[source_labels==0]\n   points=[];score_audit=[]\n   for target in CFG[\'target_fprs\']:\n    if target==.001 and len(neg)<CFG[\'minimum_benign_for_001\']:\n     points.append(dict(target_fpr=target,status=\'insufficient_source_benign_validation\',n_source_benign=len(neg)));continue\n    thr,vfpr=threshold(neg,target)\n    stable_thr,per_source=source_domain_threshold(sv,source_labels,source_domains,target)\n    oracle_thr,oracle_vfpr=oracle_threshold(st,label,target)\n    for strategy,t,source_only in [(\'pooled_source_quantile\',thr,True),(\'worst_source_quantile\',stable_thr,True),(\'target_oracle_POSTHOC\',oracle_thr,False)]:\n     metric=calc_scores(label,st,t)\n     points.append(dict(target_fpr=target,status=\'evaluated\',threshold_strategy=strategy,\n           deployable_source_only=bool(source_only),threshold=t,source_validation_fpr=float(np.mean(neg>=t)),\n           **metric))\n    score_audit.append(dict(target_fpr=target,source_pooled_threshold=thr,source_fpr=vfpr,\n       worst_source_threshold=stable_thr,per_source_thresholds=per_source,\n       target_oracle_threshold_POSTHOC=oracle_thr,oracle_target_fpr=oracle_vfpr,\n       source_only_target_fpr=calc_scores(label,st,thr)[\'test_fpr\'],\n       oracle_recall_upper_bound_POSTHOC=calc_scores(label,st,oracle_thr)[\'attack_recall\']))\n   source_platt,target_platt=source_only_platt(sv,source_labels,st)\n   calibration={\n      \'source_raw\':calibration_metrics(source_labels,sv),\n      \'target_raw_POSTHOC\':calibration_metrics(label,st),\n      \'source_platt_fit_on_source_validation\':calibration_metrics(source_labels,source_platt),\n      \'target_platt_POSTHOC\':calibration_metrics(label,target_platt),\n      \'note\':\'Platt mapping is fitted only on source validation. Target labels are used solely for evaluation. The source in-sample calibrated ECE may be optimistic; no target adaptation is performed.\'}\n   record=dict(holdout=hold,seed=seed,method=method,feature_variant=variant,feature_names=names,mode=mode,\n      source_domains=[z for z in DOMAINS if z!=hold],n_train=len(tr),n_validation=len(va),n_test=len(te),\n      n_target_benign=int(np.sum(label==0)),n_target_attacks=int(np.sum(label==1)),\n      test_sha=digest(te),validation_sha=digest(va),\n      auroc=float(roc_auc_score(label,st)),auprc=float(average_precision_score(label,st)),\n      source_auroc=float(roc_auc_score(source_labels,sv)),source_auprc=float(average_precision_score(source_labels,sv)),\n      source_distribution=score_distribution(sv,source_labels),target_distribution_POSTHOC=score_distribution(st,label),\n      threshold_audit=score_audit,calibration=calibration,**meta,operating_points=points)\n   pred_path.parent.mkdir(parents=True,exist_ok=True)\n   np.savez_compressed(pred_path,test_ids=te,test_labels=label,test_scores=st.astype(np.float32),\n      validation_ids=va,validation_labels=source_labels,validation_scores=sv.astype(np.float32),\n      source_validation_domains=source_domains,source_platt_scores=source_platt.astype(np.float32),target_platt_scores=target_platt.astype(np.float32))\n   result_path.parent.mkdir(parents=True,exist_ok=True);save_json(result_path,record)\n   if method==\'PooledLightGBM\':\n    shifted=feature_shift_diagnostics(full_x[tr],full_x[te],FEATURES,seed,hold)\n    pp=OUT/\'feature_shift\'/f\'{normalized(hold)}__{seed}.json\';pp.parent.mkdir(parents=True,exist_ok=True)\n    if not pp.exists():save_json(pp,shifted)\n   print(\'TASK_OK\',mode,worker_id,ix,\'/\',len(pending),hold,seed,method,variant,\'AUROC\',round(record[\'auroc\'],4),flush=True)\n  except Exception:\n   msg=traceback.format_exc();errors.append(dict(holdout=hold,seed=seed,method=method,variant=variant,error=msg))\n   print(\'TASK_FAILED\',hold,seed,method,variant,msg[-1400:],flush=True)\n save_json(OUT/f\'{mode}_worker_{worker_id}_failures.json\',errors)\n if errors:raise RuntimeError(f\'{len(errors)} {mode} tasks failed; inspect failures JSON\')\n\ndef gate():\n paths=list((OUT/\'records\').glob(\'pilot__*.json\')) if (OUT/\'records\').exists() else []\n if len(paths)!=len(tasks(True)):raise RuntimeError(f\'PILOT_GATE_INCOMPLETE {len(paths)}/{len(tasks(True))}\')\n data=[json.loads(p.read_text()) for p in paths]\n if any(not np.isfinite(r[\'auroc\']) for r in data):raise RuntimeError(\'PILOT_BAD_AUROC\')\n save_json(OUT/\'pilot_gate.json\',dict(status=\'passed\',model_runs=len(data),note=\'Software feasibility gate only. Pilot scores are not used for model selection or target tuning.\'))\n print(\'PILOT_GATE_PASSED\',len(data),flush=True)\n\ndef aggregate():\n records=[json.loads(p.read_text()) for p in sorted((OUT/\'records\').glob(\'main__*.json\'))]\n expected=len(tasks(False));unique={(r[\'holdout\'],r[\'seed\'],r[\'method\'],r[\'feature_variant\']) for r in records}\n if len(unique)!=expected or len(records)!=expected:raise RuntimeError(f\'INCOMPLETE_OR_DUPLICATE_MAIN_RUNS {len(unique)}/{expected}\')\n for hold in DOMAINS:\n  for seed in CFG[\'seeds\']:\n   rs=[r for r in records if r[\'holdout\']==hold and r[\'seed\']==seed]\n   if len(rs)!=len(METHODS)*len(FEATURE_VARIANTS) or len({r[\'test_sha\'] for r in rs})!=1 or len({r[\'validation_sha\'] for r in rs})!=1:\n    raise RuntimeError(\'FIXED_TEST_VALIDATION_MISMATCH \'+hold+\' \'+str(seed))\n rows=[];resources=[];threshold_rows=[];dist_rows=[];calibration_rows=[]\n for r in records:\n  h={k:r[k] for k in (\'holdout\',\'seed\',\'method\',\'feature_variant\',\'n_train\',\'n_validation\',\'n_test\',\'test_sha\',\'validation_sha\',\'auroc\',\'auprc\',\'source_auroc\',\'source_auprc\')}\n  for point in r[\'operating_points\']:\n   rows.append({**h,**point})\n   if point[\'status\']==\'evaluated\' and point[\'deployable_source_only\'] and point[\'source_validation_fpr\']>point[\'target_fpr\']+1e-12:raise RuntimeError(\'SOURCE_THRESHOLD_FPR_INVARIANT\')\n  for t in r[\'threshold_audit\']:\n   threshold_rows.append({**{k:r[k] for k in [\'holdout\',\'seed\',\'method\',\'feature_variant\']},**{k:v for k,v in t.items() if k!=\'per_source_thresholds\'},\'per_source_thresholds_json\':json.dumps(t[\'per_source_thresholds\'])})\n  for location,metrics in r[\'calibration\'].items():\n   if isinstance(metrics,dict):calibration_rows.append({**{k:r[k] for k in (\'holdout\',\'seed\',\'method\',\'feature_variant\')},\'location\':location,**metrics})\n  for prefix,dd in [(\'source\',r[\'source_distribution\']),(\'target_POSTHOC\',r[\'target_distribution_POSTHOC\'])]:\n   dist_rows.append({**{k:r[k] for k in [\'holdout\',\'seed\',\'method\',\'feature_variant\']},\'distribution\':prefix,**dd})\n  resources.append({**{k:r[k] for k in [\'holdout\',\'seed\',\'method\',\'feature_variant\',\'n_train\',\'n_validation\',\'n_test\']},**{k:r[k] for k in (\'train_seconds\',\'optimizer_steps\',\'communication_bytes\')}})\n df=pd.DataFrame(rows);df.to_csv(OUT/\'threshold_transfer_metrics.csv\',index=False)\n pd.DataFrame(threshold_rows).to_csv(OUT/\'oracle_gap_POSTHOC.csv\',index=False)\n pd.DataFrame(dist_rows).to_csv(OUT/\'score_distribution_audit.csv\',index=False)\n pd.DataFrame(calibration_rows).to_csv(OUT/\'calibration_audit.csv\',index=False)\n pd.DataFrame(resources).to_csv(OUT/\'resource_metrics.csv\',index=False)\n audit=[]\n for p in sorted((OUT/\'feature_shift\').glob(\'*.json\')):audit.extend(json.loads(p.read_text()))\n pd.DataFrame(audit).to_csv(OUT/\'feature_shift_ks.csv\',index=False)\n ev=df[df.status==\'evaluated\'];numeric=[\'attack_recall\',\'test_fpr\',\'auroc\',\'auprc\',\'attack_f1\',\'precision\',\'source_auroc\']\n summ=ev.groupby([\'holdout\',\'feature_variant\',\'method\',\'threshold_strategy\',\'target_fpr\'])[numeric].agg([\'mean\',\'std\',\'count\']).reset_index()\n summ.columns=[\'_\'.join(map(str,c)).strip(\'_\') for c in summ.columns];summ.to_csv(OUT/\'summary_mean_std.csv\',index=False)\n diffs=[]\n for (hold,seed,method,target,strategy),group in ev.groupby([\'holdout\',\'seed\',\'method\',\'target_fpr\',\'threshold_strategy\']):\n  base=group[group.feature_variant==\'all\']\n  if len(base)!=1:continue\n  for _,r in group.iterrows():\n   if r.feature_variant==\'all\':continue\n   for metric in (\'attack_recall\',\'test_fpr\',\'auroc\',\'auprc\'):\n    diffs.append(dict(holdout=hold,seed=seed,method=method,threshold_strategy=strategy,target_fpr=target,variant=r.feature_variant,metric=metric,delta=float(r[metric]-base.iloc[0][metric])))\n pd.DataFrame(diffs).to_csv(OUT/\'paired_feature_ablation_deltas.csv\',index=False)\n failures=[]\n for f in OUT.glob(\'*_worker_*_failures.json\'):failures.extend(json.loads(f.read_text()))\n if failures:raise RuntimeError(\'FAILURES_PRESENT \'+str(len(failures)))\n run_status=dict(status=\'complete\',completed_main_runs=len(records),expected_main_runs=expected,evaluated_operating_points=int(len(ev)),fixed_test_passed=True,\n    source_only_fit_and_deployable_thresholds=True,oracle_thresholds_are_posthoc_non_deployable=True,\n    feature_shift_diagnostics_posthoc_only=True,features_retrained_per_variant=True,\n    limitation=\'No capture/session independence, target labels used only for posthoc oracle and metrics, sample prevalence is balanced by design\')\n save_json(OUT/\'run_status.json\',run_status)\n archive=ROOT/CFG[\'archive\']\n with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n  for f in OUT.rglob(\'*\'):\n   if f.is_file():z.write(f,f.relative_to(ROOT))\n print(\'FINAL_ZIP\',archive,\'bytes\',archive.stat().st_size,\'runs\',expected,flush=True)\n\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'--mode\',choices=[\'prepare\',\'pilot\',\'gate\',\'main\',\'aggregate\'],required=True);p.add_argument(\'--worker\',type=int,default=0);p.add_argument(\'--workers\',type=int,default=2)\n a=p.parse_args()\n if a.mode==\'prepare\':prepare()\n elif a.mode==\'gate\':gate()\n elif a.mode==\'aggregate\':aggregate()\n else:worker(a.mode,a.worker,a.workers)\nif __name__==\'__main__\':main()\n'
engine_path.write_text(engine_source, encoding='utf-8')
print('Engine created:', engine_path, '| bytes:', engine_path.stat().st_size)
subprocess.run([sys.executable, '-m', 'py_compile', str(engine_path)], check=True)

## Automatic audit → pilot → gate → 300 main runs → ZIP

The notebook writes individual logs and JSON records for each run and can **resume completed main tasks** if rerun within the same Kaggle working directory. The data audit scans the entire attached source CSV in streaming mode. If a failure occurs, inspect `FLIDS_Experiment03_Results/*.log` and the `*_failures.json` files.

In [ ]:
import concurrent.futures, torch, time

def invoke(args, log_name):
    log_path = output / log_name
    cmd = [sys.executable, '-X', 'faulthandler', str(engine_path), *args]
    with log_path.open('w', encoding='utf-8') as handle:
        completed = subprocess.run(cmd, stdout=handle, stderr=subprocess.STDOUT, env=os.environ.copy())
    print('STEP', log_name, 'exit=', completed.returncode, flush=True)
    print(log_path.read_text(encoding='utf-8', errors='replace')[-3600:], flush=True)
    if completed.returncode:
        raise RuntimeError(f'{log_name} exited with {completed.returncode}. Read {log_path} and failure JSON files.')

def run_workers(stage, n_workers=2):
    processes = []
    for wid in range(n_workers):
        filename = output / f'{stage}_worker_{wid}.log'
        handle = filename.open('w', encoding='utf-8')
        cmd = [sys.executable, '-X', 'faulthandler', str(engine_path), '--mode', stage,
               '--worker', str(wid), '--workers', str(n_workers)]
        process = subprocess.Popen(cmd, stdout=handle, stderr=subprocess.STDOUT, env=os.environ.copy())
        processes.append((wid, process, handle, filename))
    failed=[]
    for wid, process, handle, filename in processes:
        code = process.wait()
        handle.close()
        print(stage, 'worker', wid, 'exit=', code, flush=True)
        print(filename.read_text(encoding='utf-8', errors='replace')[-2600:], flush=True)
        if code: failed.append(wid)
    if failed:
        raise RuntimeError(f'{stage} workers failed: {failed}. Inspect worker logs and *_failures.json.')

print('CUDA device count:', torch.cuda.device_count())
if not (root / 'flids_exp03_cache.npz').exists() or not (output / 'split_manifest.csv').exists():
    invoke(['--mode','prepare'], 'prepare.log')
else:
    print('RESUME: cached dataset and split manifest detected; skipping preparation.')
run_workers('pilot')
invoke(['--mode','gate'], 'pilot_gate.log')
run_workers('main')
invoke(['--mode','aggregate'], 'aggregate.log')
zip_path = root / 'FLIDS_Experiment03_Results.zip'
print('FINAL RESULTS:', zip_path, '| size MiB:', round(zip_path.stat().st_size / 2**20, 2))

In [ ]:
from IPython.display import display, FileLink
import json
status_path = output / 'run_status.json'
if status_path.exists():
    print(json.dumps(json.loads(status_path.read_text()), indent=2))
if (root / 'FLIDS_Experiment03_Results.zip').exists():
    display(FileLink(str(root / 'FLIDS_Experiment03_Results.zip'), result_html_prefix='Download Experiment 03 ZIP: '))

## Interpretation notes

Use `threshold_transfer_metrics.csv` to report operational performance and actual target FPR, keeping source-only methods separate from `target_oracle_POSTHOC`. The file `oracle_gap_POSTHOC.csv` measures recoverable threshold-transfer gaps only as a retrospective upper-bound diagnostic. Use `paired_feature_ablation_deltas.csv` for paired comparisons on identical test samples. `calibration_audit.csv` provides Brier/ECE on both source and target (target metrics are post hoc only). `feature_shift_ks.csv` provides unlabelled marginal KS scores; KS shift alone does not prove shortcut learning or causation.

All result tables should clearly state the balanced target sampling and potential sample/capture overlap limitations.